# A life-cycle model with age as a field

This notebook builds the household problem of Bardóczy, Shanker and Velásquez-Giraldo (2026, Section 2) in five Bellman stages. The within-period order is entry, mortality, productivity draw, and consumption. The terminal stage keeps the age supplied by the reader to start the backward recursion. The eight-age calibration uses selected Appendix D parameters with its own survival profile, productivity profile, and asset grid. [Federal Reserve working paper](https://www.federalreserve.gov/econres/feds/files/2026046pap.pdf)

Age is a constant field, meaning it has one value for all productivity and asset states at a given step. The consumption stage sends back the age one year younger. Mortality and productivity read their own tables at the supplied age.

Run from this notebook's folder with Bellman and NumPy installed. Familiarity with the Bellman equation and finite Markov chains is enough to read the comparison.

In [1]:
from pathlib import Path
import sys

import numpy as np

sys.path.insert(0, str(Path("../code").resolve()))

from bellman.distributions.families import label_weights, numeric_value
from bellman.refusals import ConstructionError
from bellman.trellis import connection_report

from household import build, calibration, compare, savings_grid, solve, transition_matrix
from reference import solve as solve_by_age

np.set_printoptions(precision=4, suppress=True)

## Construction and methods

The recipe names the stage files, the calibration and the methods. Construction checks the stages and joins. The earlier periods' method files state the inverse Euler equation and the endogenous grid. The policy for consumption adjacent to the terminal stage is selected once the supplied continuation has been evaluated. Calibration binds the household parameters and schedules. Configuration binds the savings points of the terminal value representation and the earlier policy grids.

In [2]:
elaborated, methodized, calibrated, configured = build()
elaborated_stages = elaborated["periods"][0]["stages"]
methodized_stages = methodized["periods"][1]["stages"]
calibrated_stages = calibrated["periods"][0]["stages"]
last = configured["periods"][0]["stages"]

print("Elaborated consumption builder signature")
print(elaborated_stages["consumption"].builder["cntn", "dcsn"].signature())
print("Methodized policy in the preceding period", methodized_stages["consumption"].methods["policy"])
print("Calibrated discount factor", calibrated_stages["consumption"].par["β"].value)
points = last["terminal"].settings["savings_grid"].value
print("Configured terminal savings points", len(points), "from", points[0], "to", points[-1])

Elaborated consumption builder signature
in         𝑉≻ : Z × Xa → ℝ
in         𝑎𝑔𝑒_next≻ : {()} → Clock
out        𝑉 : Z × Xm → ℝ
out        𝑎𝑔𝑒 : {()} → Ages
Methodized policy in the preceding period !egm   !Cartesian(z_4=Z, a_4=savings_grid)   "life-cycle:egm"
Calibrated discount factor 0.99
Configured terminal savings points 40 from 0.0 to 60.0


## Connections and terminal data

Three stages keep the age through within-period connections. The explicit connections pair the consumption stage's input **age_next** with the following entry stage's output **age**, or with the terminal stage's output **age** in the last period. The death branch retains the bequeathed assets and the realized mortality label. Only the survival branch continues to the productivity draw.

The terminal stage takes the continuation function and the constant **age**. It passes the supplied age back unchanged. Here the terminal age is 8, and the continuation function and its asset derivative are zero everywhere. The solver checks the age and requires functions for both the continuation value and its asset derivative before it constructs any numerical kernel. The terminal builder then supplies the realized grid for the value representation. The solver evaluates both terminal functions at those points before choosing the consumption method.

In [3]:
report = connection_report(configured)
assert not report.diagnostics
assert all(route.status == "certified" for route in report.routes)
print(f"{len(report.routes)} certified routes")
for route in report.routes:
    if route.source[0] in (0, 1) and route.source[1] == "consumption":
        age_pair = next(pair for pair in route.components if pair.required.endswith(":age_next"))
        print(f"{route.source} → {route.destination}, {age_pair.rule}")

terminal = last["terminal"]

def zero(z, a):
    return 0.0

terminal_inputs = {"V[>]": zero, "age": 8}
terminal_marginal = zero

try:
    terminal.check_inputs({"V[>]": zero})
except ConstructionError as error:
    print(f"Missing terminal age refused under {error.rule}")
else:
    raise AssertionError("A missing terminal age was accepted")

for requirement in terminal.check_inputs(terminal_inputs):
    print(requirement.explanation)

terminal_record = terminal.builder["dcsn", "arvl"].kernels(inputs=terminal_inputs)
print(terminal_record.grids["dcsn_to_arvl.evaluate[V[<]]"])

32 certified routes
(1, 'consumption') → (0, 'entry'), explicit connector
(0, 'consumption') → (0, 'terminal'), explicit connector
Missing terminal age refused under continuation-input
The representing function for V[>] must be measurable on its declared domain, depend only on (z_end, a_end) in that order, and take values in ℝ.
!Cartesian on (z_4, a_4), 120 points: (1, 0), (1, 0.0010114803014211297), (1, 0.008091842411369038), …, (3, 60)


## Age-specific distributions

Each forward edge supplies an age to the distribution it draws. At age 7 productivity is fixed at its previous level. At age 4 the transition matrix gives a positive probability of changing productivity. Inspect the records made from the same stage files at both ages.

In [4]:
for age in (7, 4):
    inputs = {"age": age, "V[>]": zero}
    mortality = last["mortality"].builder["dcsn", "cntn"].kernels(inputs=inputs)
    (death_law,) = mortality.laws["DistributionExtension:s"].declarations
    survival, death = [numeric_value(weight, {}) for weight in label_weights(death_law.term)]
    draw = last["draw"].builder["arvl", "dcsn"].kernels(inputs=inputs)
    (productivity_law,) = draw.laws["DistributionExtension:z"].declarations
    print(f"Age {age}, survival {survival:.3f}, death {death:.3f}")
    print(transition_matrix(productivity_law.term))

Age 7, survival 0.800, death 0.200
[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]
Age 4, survival 0.985, death 0.015
[[0.9702 0.0296 0.0002]
 [0.0148 0.9705 0.0148]
 [0.0002 0.0296 0.9702]]


## Household values and policy

The supplied terminal functions give asset-independent continuation values at each productivity and zero marginal values. The numerical representation interpolates between grid points and holds each endpoint fixed outside the savings grid. The selected explicit method therefore sets consumption equal to cash on hand. The preceding periods use the endogenous-grid method with positive discounted marginal values. Its differentiability and concavity requirements remain explicit.

The comparison evaluates the compiled value, asset derivative, budget, and savings kernels at each age. It checks those results and the forward cohort transition against a separate indexed-by-age implementation. The transition weights the saving policy by the survival and productivity probabilities, allocating mass between adjacent asset points. The reference reads the asset points directly from the settings file.

A reader with the original research script can repeat the cross-check from this folder with:

```shell
python ../code/household.py --terminal-age 8 --research-reference /path/to/check_age_as_a_field.py
```

The command checks consumption, savings, all four stage values, the arrival marginal value and the transition at every age against the arrays in that script.

In [5]:
actual, asset_points, terminal_requirements = solve(
    configured, terminal_inputs, terminal_marginal=terminal_marginal
)
terminal_policy, preceding_policy = actual[:2]
print("Policy adjacent to terminal", terminal_policy["policy_method"])
print("Policy in the preceding period", preceding_policy["policy_method"])
for requirement in preceding_policy["policy_requirements"]:
    if requirement.condition == "egm-eligibility":
        print(f"{requirement.status}: {requirement.explanation}")

reference_grid = savings_grid()
np.testing.assert_array_equal(asset_points, reference_grid)
expected = solve_by_age(calibration(), reference_grid)
differences = compare(actual, expected)
print("Ages returned", [period["age"] for period in actual])
for quantity, difference in differences.items():
    print(f"{quantity.capitalize():<12} {difference:.2e}")

Policy adjacent to terminal !explicit
Policy in the preceding period !egm   !Cartesian(z_4=Z, a_4=savings_grid)   "life-cycle:egm"
required: the construction satisfies the declared line where the continuation value 𝑉≻ is concave and continuously differentiable above the savings bound, with a positive right derivative at the bound, finite or infinite; an infinite derivative gives zero consumption as the interpolation endpoint, outside a strictly positive consumption domain
Ages returned [7, 6, 5, 4, 3, 2, 1, 0]
Consumption  1.42e-14
Savings      1.42e-14
Values       3.55e-15
Marginals    1.24e-14
Transition   1.13e-14


## Survival mass

Every row of the forward transition starts from one unit of mass at a given productivity and asset state. The row sum is the fraction that survives. Check that this holds at every age using the survival schedule from the calibration.

In [6]:
for row in actual:
    survival = calibration()["ψ"][row["age"]]
    np.testing.assert_allclose(row["transition"].sum(axis=1), survival, rtol=0, atol=1e-13)
print(f"Survival mass agrees at all {len(actual)} ages.")

Survival mass agrees at all 8 ages.


## Observations

Use this space to record any questions about the connection that supplies age, the age-specific distributions, or the policy comparison.

**Observation**

&nbsp;

**Follow-up**

&nbsp;